# EAGLE-2 Under Review: Is Context-Aware Dynamic Draft Tree Construction a Principled Primitive or Heuristic Pruning Tuned to Decoder Idiosyncrasies?

**Paper:** [https://arxiv.org/abs/2406.16858](https://arxiv.org/abs/2406.16858)  
**Repository:** [https://github.com/SafeAILab/EAGLE](https://github.com/SafeAILab/EAGLE)  
**Framework:** pytorch  
**License:** NOASSERTION  

---

*Reproduction generated by Vivory Research — runs on free-tier hardware (Kaggle T4 / Oracle CPU / GitHub Actions).*
*Produced: 2026-04-28 04:32 UTC*


## 1. Setup

Install dependencies from the paper's `requirements.txt`. Some packages may need GPU-specific wheels — adjust for your Colab/Kaggle runtime.

In [ ]:
!pip install --quiet --upgrade pip
!pip install --quiet torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install --quiet torch==2.6.0 transformers>=4.53.1 accelerate==0.26.0 fschat==0.2.31 gradio==3.50.2 openai==0.28.0 anthropic==0.5.0 sentencepiece==0.1.99 protobuf==3.19.0 wandb


## 2. Repository

Clone the reference implementation.

In [ ]:
!git clone --depth 1 https://github.com/SafeAILab/EAGLE
%cd EAGLE
!ls -la


## 3. Dataset

Download the dataset. Replace this cell with the dataset-specific loading code from the repository's README or `scripts/download_data.sh`.

In [ ]:
# TODO: Replace with dataset-specific download/load code.
# Check the repo README for instructions — common patterns:
#   bash scripts/download_data.sh
#   python -m src.data.download
#   from datasets import load_dataset; ds = load_dataset("name")
print("Dataset placeholder — fill in from repo README.")


## 4. Configuration

Core hyperparameters. Consider reducing epochs/batch size to fit free-tier GPU limits (Kaggle T4: 16GB VRAM, 30h/week; Colab: variable).

In [ ]:
import os, json, random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Reduced for free-tier — adjust if you have more GPU budget.
CONFIG = {
    "seed": SEED,
    "max_epochs": 1,
    "batch_size": 16,
    "learning_rate": 1e-4,
    "subset_fraction": 0.1,  # use 10% of data for quick reproduction
}
print(json.dumps(CONFIG, indent=2))


## 5+6. Paper-aware evaluation (auto-generated)

The cell below was generated by Vivory's reproduction agent (Opus 4.7) from the paper's abstract, body, repo README, and claimed_metrics. It performs real measurement on a small subset and writes the result to `/kaggle/working/metrics.json` for the runner to ingest.

In [ ]:
import os, sys, json, time, subprocess, traceback

os.makedirs("/kaggle/working", exist_ok=True)
METRICS_PATH = "/kaggle/working/metrics.json"

def write_unsupported(reason):
    print(f"[UNSUPPORTED] {reason}")
    with open(METRICS_PATH, "w") as f:
        json.dump({"unsupported_infrastructure": 1.0}, f)

try:
    repo_dir = None
    for d in ["EAGLE", "eagle"]:
        if os.path.isdir(d):
            repo_dir = os.path.abspath(d); break
    if repo_dir is None:
        subprocess.run(["git", "clone", "--depth=1", "https://github.com/SafeAILab/EAGLE.git"], check=True)
        repo_dir = os.path.abspath("EAGLE")
    os.chdir(repo_dir)
    sys.path.insert(0, repo_dir)

    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "fschat", "accelerate", "sentencepiece", "protobuf"], check=False)

    import torch
    if not torch.cuda.is_available():
        write_unsupported("CUDA not available"); sys.exit(0)
    free_gb = torch.cuda.mem_get_info()[0] / 1e9
    print(f"Free GPU memory: {free_gb:.2f} GB")
    if free_gb < 13.0:
        write_unsupported(f"Need ~13GB for Vicuna-7B fp16, have {free_gb:.1f}GB"); sys.exit(0)

    from eagle.model.ea_model import EaModel
    from fastchat.model import get_conversation_template

    BASE = "lmsys/vicuna-7b-v1.3"
    EA = "yuhuili/EAGLE-Vicuna-7B-v1.3"

    print("Loading models...")
    model = EaModel.from_pretrained(
        base_model_path=BASE,
        ea_model_path=EA,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
        device_map="cuda:0",
        total_token=-1,
    )
    model.eval()
    tokenizer = model.get_tokenizer()

    prompts = [
        "Compose an engaging travel blog post about a recent trip to Hawaii, highlighting cultural experiences and must-see attractions.",
        "Write a short story about a detective who solves crimes using machine learning.",
        "Explain how photosynthesis works in plants, step by step.",
        "Implement a Python function that returns the n-th Fibonacci number using dynamic programming.",
        "Describe the differences between supervised and unsupervised learning with concrete examples.",
    ]

    def build_input(prompt):
        conv = get_conversation_template("vicuna")
        conv.append_message(conv.roles[0], prompt)
        conv.append_message(conv.roles[1], None)
        text = conv.get_prompt()
        return tokenizer(text, return_tensors="pt").input_ids.to("cuda")

    MAX_NEW = 128

    @torch.no_grad()
    def vanilla_run(input_ids):
        torch.cuda.synchronize(); t0 = time.time()
        out = model.base_model.generate(
            input_ids=input_ids, max_new_tokens=MAX_NEW,
            do_sample=False, pad_token_id=tokenizer.eos_token_id)
        torch.cuda.synchronize(); dt = time.time() - t0
        n = out.shape[1] - input_ids.shape[1]
        return n, dt

    @torch.no_grad()
    def eagle_run(input_ids, total_token):
        model.total_token = total_token
        torch.cuda.synchronize(); t0 = time.time()
        out = model.eagenerate(
            input_ids, temperature=0.0, max_new_tokens=MAX_NEW, log=False)
        torch.cuda.synchronize(); dt = time.time() - t0
        n = out.shape[1] - input_ids.shape[1]
        return n, dt

    print("Warmup...")
    wid = build_input("Hello")
    _ = eagle_run(wid, 60); _ = eagle_run(wid, 26); _ = vanilla_run(wid)

    eagle1_speeds, eagle2_speeds, vanilla_speeds = [], [], []
    for i, p in enumerate(prompts):
        ids = build_input(p)
        nv, tv = vanilla_run(ids)
        n2, t2 = eagle_run(ids, 60)
        n1, t1 = eagle_run(ids, 26)
        v_s = nv / tv; e1_s = n1 / t1; e2_s = n2 / t2
        vanilla_speeds.append(v_s); eagle1_speeds.append(e1_s); eagle2_speeds.append(e2_s)
        print(f"[{i+1}] vanilla={v_s:.2f}t/s  eagle1={e1_s:.2f}t/s  eagle2={e2_s:.2f}t/s")

    e2_ratios = [e2 / v for e2, v in zip(eagle2_speeds, vanilla_speeds)]
    e1_ratios = [e1 / v for e1, v in zip(eagle1_speeds, vanilla_speeds)]
    improvements = [(r2 / r1 - 1.0) * 100.0 for r1, r2 in zip(e1_ratios, e2_ratios)]

    metrics = {
        "speedup_ratio_min": float(min(e2_ratios)),
        "speedup_ratio_max": float(max(e2_ratios)),
        "speedup_improvement_over_eagle1_min": float(min(improvements)),
        "speedup_improvement_over_eagle1_max": float(max(improvements)),
    }
    with open(METRICS_PATH, "w") as f:
        json.dump(metrics, f, indent=2)
    print(json.dumps(metrics, indent=2))

except Exception as e:
    traceback.print_exc()
    write_unsupported(f"{type(e).__name__}: {e}")

## Appendix — Reproduction policy

This notebook runs on **free-tier hardware only**:

- **Kaggle Notebooks** — T4 GPU, 30h/week quota
- **Oracle Cloud** — ARM 4-core CPU, no GPU
- **GitHub Actions** — 2-core CPU, no GPU, 6h timeout
- **Colab** — variable T4/V100, 12h sessions (manual only)

If the full experiment exceeds these limits, reduce `max_epochs` / `subset_fraction` in the config cell and note the delta in the reproduction report.

**Paper estimates:**  ~48.0h runtime  
